# Approach 5 — CharNet Training (current approach)

**Before running:** Runtime → Change runtime type → **T4 GPU**

### What this trains
`CharNet`: character index (0–61) + category embedding → 6 cubic Bézier curves (48 values).  
No reference image, no writer ID — just one integer in, stroke curves out.  
Natural pen variation is added at generation time via noise.

### First run note
The first training run builds `bezier_labels.npy` by skeletonising all handwriting images (~5 min).  
Subsequent runs load from cache instantly.

### After training
Download `style_net.pt` and place it at:  
`approaches/05_CharNet/checkpoints/style_net.pt`  
Then run locally: `cd approaches/05_CharNet && python run.py`

In [ ]:
# Cell 1: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Install dependencies
!pip install scipy scikit-image -q
print('Dependencies installed')

In [ ]:
# Cell 4: Clone repo
import os

REPO      = 'AI-Powered-Handwriting-Generation-System'
REPO_PATH = f'/content/{REPO}'

if os.path.exists(REPO_PATH):
    print('Repo exists — pulling latest...')
    os.system(f'git -C {REPO_PATH} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone --depth 1 https://github.com/Abdullahshaz70/{REPO}.git {REPO_PATH}')

APPROACH_DIR = f'{REPO_PATH}/approaches/05_CharNet'
print('Approach dir:', APPROACH_DIR)

In [ ]:
# Cell 5: Configure training
EPOCHS = 300   # 200-300 recommended; fast training since no image loading per batch

print(f'Will train for {EPOCHS} epochs')
print('Note: first run builds bezier_labels.npy cache — takes ~5 minutes')

In [ ]:
# Cell 6: Train
os.chdir(APPROACH_DIR)
!python run.py --train --epochs {EPOCHS}

In [ ]:
# Cell 7: Save checkpoint to Drive + download
import shutil
from google.colab import files

ckpt_src  = f'{APPROACH_DIR}/checkpoints/style_net.pt'
drive_dir = '/content/drive/MyDrive/handwriting_checkpoints/05_CharNet'
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(ckpt_src, os.path.join(drive_dir, 'style_net.pt'))
print('Saved to Drive:', drive_dir)

files.download(ckpt_src)
print('\nDownload started.')
print('Place style_net.pt in:  approaches/05_CharNet/checkpoints/')
print('Then run locally:       cd approaches/05_CharNet && python run.py')

In [ ]:
# Cell 8: Preview — generate full alphabet
import sys
sys.path.insert(0, f'{APPROACH_DIR}/src')

import torch
import numpy as np
import matplotlib.pyplot as plt

from generate import load_model, generate_char
from data     import CHAR_TO_IDX, CANVAS_SIZE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, ckpt, device = load_model(f'{APPROACH_DIR}/checkpoints/style_net.pt', device)
print(f'Loaded: epoch={ckpt["epoch"]}, val_MSE={ckpt["val_loss"]:.6f}')

groups = [
    ('a-z', list('abcdefghijklmnopqrstuvwxyz')),
    ('A-Z', list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')),
    ('0-9', list('0123456789')),
]

fig, axes = plt.subplots(3, 26, figsize=(26, 4))
for row, (label, chars) in enumerate(groups):
    for col in range(26):
        ax = axes[row][col]
        if col < len(chars) and chars[col] in CHAR_TO_IDX:
            arr = generate_char(model, chars[col], device)
            ax.imshow(arr, cmap='gray', vmin=0, vmax=255)
            ax.set_title(chars[col], fontsize=7)
        ax.axis('off')

plt.suptitle(f'CharNet — {EPOCHS} epochs  (val_MSE={ckpt["val_loss"]:.6f})')
plt.tight_layout()
plt.show()